# Notebook 01 — Data Exploration

**Project:** Machine Learning-Based Intrusion Detection for Cloud Network Security: A Zero Trust Architecture Approach  
**Author:** Dingaan Mahlatse Machethe  
**Institution:** EC-Council University | ECCU500: Managing Secure Network Systems  

---

## Objectives
- Load and inspect the NSL-KDD dataset (148,517 records, 41 features)
- Understand attack category distribution (DoS, Probe, R2L, U2R)
- Examine class imbalance — justifies SMOTE in preprocessing
- Profile feature distributions and identify categorical vs numerical features
- Document dataset characteristics matching paper Appendix A (Table A1)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.preprocess import download_nsl_kdd, NSL_KDD_COLUMNS, CATEGORICAL_COLS, ATTACK_LABEL_MAP

sns.set_theme(style='whitegrid', palette='husl')
print('Libraries loaded.')

## 1. Load NSL-KDD Dataset

In [ ]:
df = download_nsl_kdd()
print(f'Shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')
df.head()

## 2. Dataset Statistics (Paper Appendix A — Table A1)

In [ ]:
print('=== NSL-KDD Dataset Characteristics (Paper Appendix A) ===')
print(f'Total Records   : {len(df):,}  (paper: 148,517)')
print(f'Total Features  : {len(df.columns) - 1}  (paper: 41)')
print(f'Label Column    : label')
print(f'Attack Categories: 4 (DoS, Probe, R2L, U2R)')
print()
print('--- Label Value Counts ---')
print(df['label'].value_counts().head(20))

## 3. Attack Category Distribution

In [ ]:
# Map labels to 5-class (normal + 4 attack types)
dos_attacks = ['back','land','neptune','pod','smurf','teardrop','apache2','udpstorm','processtable','worm','mailbomb']
probe_attacks = ['ipsweep','nmap','portsweep','satan','mscan','saint']
r2l_attacks = ['ftp_write','guess_passwd','imap','multihop','phf','spy','warezclient','warezmaster','sendmail','named','snmpgetattack','snmpguess','xlock','xsnoop','httptunnel']
u2r_attacks = ['buffer_overflow','loadmodule','perl','rootkit','ps','sqlattack','xterm']

def categorise_attack(label):
    label = label.strip().lower()
    if label == 'normal': return 'Normal'
    if label in dos_attacks: return 'DoS'
    if label in probe_attacks: return 'Probe'
    if label in r2l_attacks: return 'R2L'
    if label in u2r_attacks: return 'U2R'
    return 'Other'

df['category'] = df['label'].apply(categorise_attack)
cat_counts = df['category'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cat_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71','#e74c3c','#3498db','#f39c12','#9b59b6'], edgecolor='white')
axes[0].set_title('Attack Category Distribution\n(NSL-KDD Training Set)', fontweight='bold')
axes[0].set_xlabel('Category'); axes[0].set_ylabel('Count'); axes[0].tick_params(rotation=0)
for bar, count in zip(axes[0].patches, cat_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200, f'{count:,}', ha='center', fontsize=9)

cat_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#2ecc71','#e74c3c','#3498db','#f39c12','#9b59b6'])
axes[1].set_title('Proportion by Category', fontweight='bold'); axes[1].set_ylabel('')
plt.suptitle('NSL-KDD Dataset — 4 Attack Categories + Normal\nRef: Paper §5.2, Appendix A', fontsize=11)
plt.tight_layout(); plt.savefig('../results/figures/01_attack_categories.png', dpi=150, bbox_inches='tight')
plt.show()
print(cat_counts)

## 4. Class Imbalance Analysis (Motivates SMOTE)

In [ ]:
binary_dist = df['category'].map(lambda x: 'Normal' if x == 'Normal' else 'Attack').value_counts()
normal_pct = binary_dist['Normal'] / len(df) * 100
attack_pct = binary_dist['Attack'] / len(df) * 100
print(f'Normal : {binary_dist["Normal"]:,} ({normal_pct:.2f}%)  — paper: 53.46%')
print(f'Attack : {binary_dist["Attack"]:,} ({attack_pct:.2f}%)  — paper: 46.54%')
print('\nNote: SMOTE applied in preprocessing to balance classes (src/preprocess.py Step 5)')

## 5. Categorical Feature Inspection (OHE candidates)

In [ ]:
for col in CATEGORICAL_COLS:
    print(f'\n--- {col} ({df[col].nunique()} unique values) ---')
    print(df[col].value_counts().head(10))

## 6. Numerical Feature Distributions

In [ ]:
numeric_cols = [c for c in df.columns if c not in CATEGORICAL_COLS + ['label', 'category']]
print(f'Numerical features: {len(numeric_cols)}')
df[numeric_cols].describe().T[['mean','std','min','max']].sort_values('std', ascending=False).head(15)

In [ ]:
# Top 4 most-variable features — visualise by attack category
top_features = ['src_bytes', 'dst_bytes', 'count', 'srv_count']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flatten(), top_features):
    data = df[df[feat] < df[feat].quantile(0.99)]  # clip outliers for viz
    for cat in ['Normal','DoS','Probe']:
        subset = data[data['category'] == cat][feat]
        if len(subset) > 0:
            ax.hist(subset, bins=50, alpha=0.5, label=cat, density=True)
    ax.set_title(feat); ax.legend(fontsize=8)
plt.suptitle('Feature Distributions by Attack Category (top 1% clipped)', fontsize=11)
plt.tight_layout(); plt.savefig('../results/figures/01_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary

| Metric | Value | Paper Reference |
|--------|-------|-----------------|
| Total records | 148,517 | Appendix A, Table A1 |
| Features | 41 | Appendix A, Table A1 |
| Attack categories | 4 (DoS, Probe, R2L, U2R) | §5.2 |
| Normal:Attack ratio | 53.46:46.54 | Appendix A, Table A1 |
| Categorical cols | 3 (protocol_type, service, flag) | §5.2 |

**Next:** `02_preprocessing.ipynb` — apply the full pipeline (OHE → MinMaxScaler → SMOTE → RFECV → split)